In [1]:
import re
import numpy as np

In [2]:
file_name = 'CBOW.txt'

In [4]:
with open(file_name,"r",encoding='utf-8') as f:
  corpus = f.read().lower()

In [5]:
corpus

'the speed of transmission is an important point of difference between the two viruses. influenza has a shorter median incubation period (the time from infection to appearance of symptoms) and a shorter serial interval (the time between successive cases) than covid-19 virus. the serial interval for covid-19 virus is estimated to be 5-6 days, while for influenza virus, the serial interval is 3 days. this means that influenza can spread faster than covid-19. \n\nfurther, transmission in the first 3-5 days of illness, or potentially pre-symptomatic transmission –transmission of the virus before the appearance of symptoms – is a major driver of transmission for influenza. in contrast, while we are learning that there are people who can shed covid-19 virus 24-48 hours prior to symptom onset, at present, this does not appear to be a major driver of transmission. \n\nthe reproductive number – the number of secondary infections generated from one infected individual – is understood to be betwe

In [7]:
#tokenize
tokens = re.findall(r"\b\w+\b",corpus)

In [14]:
#create vocab
vocab = sorted(set(tokens))
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}
V = len(vocab)

In [15]:
#one_hot encoding
def one_hot(index,V):
  vec= np.zeros(V)
  vec[index] = 1
  return vec

In [16]:
X,Y=[],[]
window_size = 2
for i,word in enumerate(tokens):
  context=[]
  for j in range (i-window_size,i+window_size+1):
    if j!=i and j>=0 and j<len(tokens):
      context.append(tokens[j])
  if len(context) ==0:
    continue

  context_vec = np.mean([one_hot(word2idx[w],V) for w in context],axis =0)
  X.append(context_vec)
  Y.append(one_hot(word2idx[word],V))

In [29]:
X = np.array(X)
Y = np.array(Y)

In [25]:
np.random.seed(42)
embed_dim  =50
lr = 0.05
epochs = 200

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Dense
from tensorflow.keras.optimizers import Adam

In [27]:
model  = Sequential([
    Input(shape=(V,)),
    Dense(embed_dim,activation='relu'),
    Dense(V,activation='softmax')
])

model.compile(
    optimizer= Adam(learning_rate=0.05),
    metrics=['accuracy'],
    loss='categorical_crossentropy'
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 100)            │         5,100 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,150 (39.65 KB)

 Trainable params: 10,150 (39.65 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
history = model.fit(X,Y, epochs = 200,verbose =0)

In [36]:
model.layers[0].get_weights()[0]

array([[ 1.2258832 , -0.28949317,  0.6890019 , ...,  2.0312083 ,
         0.3243634 , -0.08292407],
       [ 1.7725625 , -0.7832847 , -0.50989604, ..., -0.4880521 ,
         0.11478506, -0.45025688],
       [ 0.0565031 , -0.03138657,  1.7048918 , ...,  0.19291753,
        -0.23318885, -0.50700307],
       ...,
       [ 1.830375  ,  1.6996484 , -1.0628031 , ..., -0.75630844,
        -0.14952926,  0.23702891],
       [ 0.33416355, -1.5839003 , -0.3716057 , ...,  1.2327712 ,
        -0.16375002,  1.4554635 ],
       [ 1.1599555 , -0.47562602, -1.4585918 , ...,  1.290902  ,
         0.10212345,  1.8041531 ]], dtype=float32)

In [34]:
embeddings = model.layers[0].get_weights()[0]
#normalize the embeddings
embeddings = embeddings / (np.linalg.norm(embeddings,keepdims=True,axis =1)+1e-9)

In [39]:
def predict_target(context_words):
  valid_words = [w for w in context_words if w in word2idx]
  if not valid_words:
    print("None of the words match the corpus")
    return None

  context_vec = np.mean([one_hot(word2idx[w],V)for w in valid_words],axis=0)
  context_vec = context_vec.reshape(1,-1)

  prediction  = model.predict(context_vec,verbose=0)[0]
  target_index = np.argmax(prediction)
  target_word = idx2word[target_index]

  return target_word


In [42]:
def nearest_similar(word,k):
  if word not in word2idx:
    return[]

  i = word2idx[word]
  vec = embeddings[i]

  sims = np.dot(embeddings,vec)
  sims = sims/(np.linalg.norm(embeddings,keepdims=True,axis=1)+1e-9)

  top_k = np.argsort(-sims)[1:k+1]
  return [(idx2word[j],float(sims[j])) for j in top_k]

In [45]:
Input_context_words = (['the' ,'time','successive', 'cases'])
target_word = predict_target(Input_context_words)
print(target_word)

between
